In [ ]:
import torch
import json
import pandas as pd
import matplotlib.pyplot as plt

from torch_geometric.loader import DataLoader
from torch_geometric import explain
from torch_geometric.data import Data

from fgnn.models import get_model
from fgnn.data import get_data
from fgnn.utils.utils import DotDict

import lovely_tensors as lt
lt.monkey_patch()

import matplotlib.cm as cm
import matplotlib.colors as mcolors

In [ ]:
dataset_params = dict(
    path="Data/Water_Graph_dist.pt",
    window_size=12,
    stride=0.3,
    fuzzy=dict(
        n_clusters=3,
    ),
    return_raw=True,
)

class FakeLogger:
    def info(self, msg):
        print(msg)
logger = FakeLogger()

train_data, val_data, test_data, raw_data = get_data(dataset_params, logger)

In [ ]:
node_dim = train_data[0].x.shape[1]
edge_dim = None
num_classes = 1
node_names = raw_data.node_names
pipe_names = raw_data.pipe_names

In [ ]:
model_params = dict(
    name="GNN",
    hidden_dim=128,
    latent_dim=32,
    depth=8,
    dropout=0.3,
    in_channels=node_dim,
    num_classes=num_classes
)
model_params = DotDict(model_params)
ckpt_path = "checkpoints/best_model_2tee09zr.pth"

model = get_model(model_params)
model.eval()

model.load_state_dict(torch.load(ckpt_path))

In [ ]:
class ExplainWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x, edge_index):
        return self.model(Data(x=x, edge_index=edge_index))

In [ ]:
model = ExplainWrapper(model)

In [ ]:
leak_data = [
    test_data[i] for i in range(len(test_data)) if test_data[i].edge_label.sum()
]
batch_size = 1
test_batch = next(
        iter(DataLoader(leak_data, batch_size=batch_size, shuffle=False))
    )

In [ ]:
model_config = explain.ModelConfig(
    mode="binary_classification",
    task_level="node",
    return_type="raw",
)

explainer = explain.Explainer(
    model=model,
    algorithm=explain.GNNExplainer(),
    explanation_type="model",
    model_config=model_config,
    node_mask_type="attributes",
    edge_mask_type="object",
)

# explainer = explain.Explainer(
#     model=model,
#     algorithm=explain.PGExplainer(epochs=30, lr=0.003),
#     explanation_type='phenomenon',
#     edge_mask_type='object',
#     model_config=model_config,
# )

# # Train against a variety of node-level or graph-level predictions:
# for epoch in range(30):
#     print(f"Epoch {epoch}")
#     for data in train_data:
#         x, edge_index = data.x, data.edge_index
#         # Explain a leak node:
#         leak_nodes = torch.where(data.edge_label == 1)[0]
#         if len(leak_nodes) == 0:
#             continue
#         else:
#             leak_nodes = leak_nodes[0]
#         loss = explainer.algorithm.train(epoch, model, x, edge_index, index=leak_nodes, target=1)

In [ ]:
leak_edges = list(torch.where(test_batch.edge_label))[0]
leak_edge = None
print(f"Leak edges: {leak_edges.p}")

In [ ]:
leak_edge = leak_edges[0]
print(f"Explaining leak in edge {leak_edge}")

In [ ]:
explanation = explainer(x=test_batch.x, edge_index=test_batch.edge_index, index=leak_edge)
explanation.validate()

In [ ]:
# load coordinates

coords_df = pd.read_excel("Data/static2.xlsx", index_col=0)

coord_lookup = {
    node: (coords_df.loc[node, "X-Coord"], coords_df.loc[node, "Y-Coord"])
    for node in node_names
}

In [ ]:
edge_index = explanation.edge_index
edge_mask = explanation.edge_mask

# Normalize for visualization only
edge_importance = edge_mask.detach().cpu()
edge_importance = (edge_importance - edge_importance.min()) / (
    edge_importance.max() - edge_importance.min() + 1e-8
)


In [ ]:
from pyvis.network import Network

viridis = cm.get_cmap("viridis")

def rgba_from_cmap(cmap, value, alpha):
    r, g, b, _ = cmap(value)
    return f"rgba({int(r*255)},{int(g*255)},{int(b*255)},{alpha})"


In [ ]:
from pyvis.network import Network

def visualize_full_graph_geometric(
    edge_index,
    edge_importance,
    edge_labels,
    explanation_edge,
    node_names,
    coord_lookup,
    directed=False,
):
    net = Network(
        height="900px",
        width="100%",
        bgcolor="#ffffff",
        font_color="black",
        directed=directed,
        notebook=True,
    )

    # Coordinates are ground truth
    net.toggle_physics(False)

    # --- Nodes ---
    for i, node in enumerate(node_names):
        x, y = coord_lookup[node]
        net.add_node(
            i,
            label=node,
            x=float(x),
            y=-float(y),
            size=10,
            physics=False,
        )

    # --- Edges ---
    ei = edge_index.t().cpu().tolist()
    w = edge_importance.cpu().tolist()
    labels = edge_labels.detach().cpu().tolist()

    for i, ((u, v), imp, lbl) in enumerate(zip(ei, w, labels)):
        alpha = 0.15 + 0.85 * imp
        width = 1 + 4 * imp

        if i == explanation_edge:  # edge to explain
            color = "rgba(255,0,255," + str(alpha) + ")"
        if lbl == 1:   # leaking (red)
            color = "rgba(255,0,0," + str(alpha) + ")"
        else:          # non-leaking
            color = rgba_from_cmap(viridis, imp, alpha)

        net.add_edge(
            int(u),
            int(v),
            value=width,
            color=color,
            smooth=False,
        )

    return net


In [ ]:
edge_importance_thresh = 0.6

edge_importance_masked = edge_importance.clone()
edge_importance_masked[edge_importance < edge_importance_thresh] = 0.

In [ ]:
net = visualize_full_graph_geometric(
    edge_index=edge_index,
    edge_importance=edge_importance_masked,
    explanation_edge=leak_edge,
    edge_labels=test_batch.edge_label,
    node_names=node_names,
    coord_lookup=coord_lookup,
    directed=False,
)


In [ ]:
net.show("index.html")